# RAG Colab Notebook
This single notebook runs in Google Colab. It installs dependencies and provides an interactive `ipywidgets` UI to upload a PDF, choose a domain (Media, Law, Telecom, General), and either ask questions or generate a structured summary using a Retrieval-Augmented Generation (RAG) pipeline.

Run the first code cell to install dependencies and set your `OPENAI_API_KEY` value (in Cell 1). Then run the second cell to show the UI.

In [ ]:
# Colab Cell 1: Environment setup & OpenAI key
# Run this cell first. It installs packages and defines the OpenAI API key constant used by the notebook.

# Install required packages with Colab-compatible versions to avoid dependency conflicts
!pip install -q pypdf langchain langchain-community langchain-openai chromadb ipywidgets sentence-transformers requests==2.32.4 opentelemetry-api==1.38.0 opentelemetry-proto==1.38.0 opentelemetry-exporter-otlp-proto-common==1.38.0 opentelemetry-sdk==1.38.0

# Enable ipywidgets in Colab (may need refresh)
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except Exception:
    pass

# Imports
import os
import json
import subprocess
import time
from uuid import uuid4

# -- CONFIG: set your OpenAI API key here --
# Replace the empty string with your OpenAI key before running the UI cell below.
OPENAI_API_KEY = ""

# Print installed package versions to verify compatibility
try:
    import importlib.metadata as metadata
except ImportError:
    import importlib_metadata as metadata

for package in [
    "requests",
    "opentelemetry-api",
    "opentelemetry-proto",
    "opentelemetry-exporter-otlp-proto-common",
    "opentelemetry-sdk",
    "langchain",
    "chromadb",
    "ipywidgets",
    "pypdf",
    "sentence-transformers",
]:
    try:
        print(f"{package}=={metadata.version(package)}")
    except Exception as e:
        print(f"{package} not available: {e}")

print("Install finished. Set `OPENAI_API_KEY` in Cell 1, then run the UI cell.")

In [ ]:
# Colab Cell 2: RAG app with ipywidgets UI
# Run after Cell 1. The UI allows uploading a PDF, asking questions, and generating summaries.

# Imports for core logic
import os
import io
import threading
from pypdf import PdfReader
from IPython.display import display, Markdown
import ipywidgets as widgets
from uuid import uuid4

# LangChain & vectorstore imports
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain.embeddings import OpenAIEmbeddings
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.llms import OpenAI as LangOpenAI

# Utility: read PDF bytes -> text
def pdf_bytes_to_text(pdf_bytes):
    reader = PdfReader(io.BytesIO(pdf_bytes))
    text_pages = []
    for p in reader.pages:
        try:
            txt = p.extract_text() or ""
        except Exception:
            txt = ""
        text_pages.append(txt)
    return "\n\n".join(text_pages)

# Build vectorstore from PDF bytes, returns Chroma retriever
def build_vectorstore_from_pdf(pdf_bytes, persist_dir=None):
    """Build a Chroma vectorstore using OpenAI embeddings (read from Cell 1 variable or env)."""
    openai_key = globals().get("OPENAI_API_KEY") or os.getenv("OPENAI_API_KEY", "")

    # Extract text and split
    text = pdf_bytes_to_text(pdf_bytes)
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    docs = splitter.split_text(text)

    # Use OpenAI embeddings only
    embedding_fn = OpenAIEmbeddings(model="text-embedding-3-small", openai_api_key=openai_key)

    # Persist directory or temp
    persist_dir = persist_dir or f"chroma_store_{uuid4().hex}"
    vectordb = Chroma.from_texts(documents=docs, embedding=embedding_fn, persist_directory=persist_dir)
    return vectordb.as_retriever(search_kwargs={"k": 4}), persist_dir

# LLM getter
def get_llm():
    openai_key = globals().get("OPENAI_API_KEY") or os.getenv("OPENAI_API_KEY", "")
    return LangOpenAI(model_name="gpt-4o-mini", openai_api_key=openai_key, temperature=0.0)

# Domain-aware prompt template enforcing strict grounding
BASE_PROMPT = """
You are a helpful assistant that strictly answers only from the provided CONTEXT. Use the CONTEXT snippets to answer the QUESTION.
Domain: {domain}

RULES:
- Use only the provided CONTEXT to answer.
- If the answer cannot be found verbatim or inferred from the CONTEXT, respond exactly: "I cannot find that information in the provided document."
- Keep answers concise and focused.
- When the domain is "Law", emphasize definitions, clauses, and liabilities.
- When the domain is "Telecom" or "Media", emphasize technical specs, service terms, or metrics.
- When the domain is "General", provide concise, neutral answers.

CONTEXT:
{context}

QUESTION: {question}

Answer:
"""

PROMPT_TEMPLATE = PromptTemplate(template=BASE_PROMPT, input_variables=["context", "question", "domain"])

# Query function
def query_with_rag(retriever, question, domain):
    llm = get_llm()
    qa_chain = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever, chain_type_kwargs={"prompt": PROMPT_TEMPLATE})
    result = qa_chain.run({"query": question, "domain": domain})
    if "I cannot find that information in the provided document." in result:
        return "I cannot find that information in the provided document."
    return result

# Summary function
SUMMARY_PROMPT = """
You are a summarization assistant. Use only the CONTEXT below to produce a structured summary.
Domain: {domain}

If important details are missing in CONTEXT, state "I cannot find that information in the provided document."
Context:
{context}

Produce a concise structured summary with headings where useful.
"""
SUMMARY_TEMPLATE = PromptTemplate(template=SUMMARY_PROMPT, input_variables=["context", "domain"])

def generate_summary(retriever, domain):
    llm = get_llm()
    docs = retriever.get_relevant_documents("summary")
    combined = "\n\n".join([d.page_content for d in docs])
    prompt = SUMMARY_TEMPLATE.format(context=combined, domain=domain)
    llm_response = llm(prompt)
    if "I cannot find that information in the provided document." in llm_response:
        return "I cannot find that information in the provided document."
    return llm_response

# -----------------------------
# UI using ipywidgets
# -----------------------------
# Widgets (OpenAI-only)
domain_dropdown = widgets.Dropdown(options=["Media", "Law", "Telecom", "General"], value="General", description="Domain:")
file_uploader = widgets.FileUpload(accept=".pdf", multiple=False, description="Upload PDF")

ask_text = widgets.Text(value="", description="Ask a Question:", layout=widgets.Layout(width="70%"))
ask_button = widgets.Button(description="Ask", button_style="primary")
summary_button = widgets.Button(description="Generate Summary", button_style="info")
status_out = widgets.Output(layout=widgets.Layout(border="1px solid #ccc", padding="10px"))
console_out = widgets.Output(layout=widgets.Layout(border="1px solid #444", padding="10px"))

# State holders
_state = {"retriever": None, "persist_dir": None, "pdf_filename": None}

# Handlers
def on_upload_and_index():
    if len(file_uploader.value) == 0:
        with status_out:
            status_out.clear_output()
            print("Please upload a PDF first.")
        return
    uploaded = next(iter(file_uploader.value.values()))
    fname = uploaded["metadata"]["name"]
    b = uploaded["content"]
    with status_out:
        status_out.clear_output()
        print(f"Indexing {fname} ... This may take a moment.")
    def worker():
        try:
            retriever, persist_dir = build_vectorstore_from_pdf(b)
            _state["retriever"] = retriever
            _state["persist_dir"] = persist_dir
            _state["pdf_filename"] = fname
            with status_out:
                status_out.clear_output()
                print(f"Indexed '{fname}' into Chroma at {persist_dir}. Ready to query.")
        except Exception as e:
            with status_out:
                status_out.clear_output()
                print("Indexing failed:", e)
    threading.Thread(target=worker).start()

# Automatically index when file uploaded
def on_file_upload_change(change):
    if change['new']:
        on_upload_and_index()

file_uploader.observe(on_file_upload_change, names='value')

def display_answer(text):
    console_out.clear_output()
    with console_out:
        display(Markdown(text))

def on_ask_clicked(b):
    if _state["retriever"] is None:
        with status_out:
            status_out.clear_output()
            print("No document indexed. Upload a PDF first.")
        return
    question = ask_text.value.strip()
    if not question:
        with status_out:
            status_out.clear_output()
            print("Please type a question.")
        return

    domain = domain_dropdown.value
    with status_out:
        status_out.clear_output()
        print("Processing question...")

    def worker():
        try:
            ans = query_with_rag(_state["retriever"], question, domain)
            display_answer(ans)
            with status_out:
                status_out.clear_output()
                print("Done.")
        except Exception as e:
            with status_out:
                status_out.clear_output()
                print("Query failed:", e)
    threading.Thread(target=worker).start()

ask_button.on_click(on_ask_clicked)

def on_summary_clicked(b):
    if _state["retriever"] is None:
        with status_out:
            status_out.clear_output()
            print("No document indexed. Upload a PDF first.")
        return
    domain = domain_dropdown.value
    with status_out:
        status_out.clear_output()
        print("Generating summary...")

    def worker():
        try:
            summ = generate_summary(_state["retriever"], domain)
            display_answer(summ)
            with status_out:
                status_out.clear_output()
                print("Summary generated.")
        except Exception as e:
            with status_out:
                status_out.clear_output()
                print("Summary failed:", e)
    threading.Thread(target=worker).start()

summary_button.on_click(on_summary_clicked)

# Layout composition
config_box = widgets.VBox([
    widgets.HTML("<b>Configuration</b>"),
    widgets.HTML("Using OpenAI. Set `OPENAI_API_KEY` in Cell 1."),
])

upload_box = widgets.VBox([
    widgets.HTML("<b>Upload Document</b>"),
    domain_dropdown,
    file_uploader
])

action_box = widgets.VBox([
    widgets.HTML("<b>Actions</b>"),
    widgets.HBox([ask_text, ask_button]),
    widgets.HTML("<i>Or</i>"),
    summary_button
])

ui = widgets.VBox([
    widgets.HBox([config_box, upload_box, action_box]),
    widgets.HTML("<hr>"),
    widgets.HTML("<b>Status</b>"),
    status_out,
    widgets.HTML("<b>Console Output</b>"),
    console_out
])

display(ui)
print("UI ready. Set `OPENAI_API_KEY` in Cell 1, upload a PDF, then Ask or Generate Summary.")
